In [ ]:
import os
from google.colab import drive, userdata

# Secrets do Colab (ícone da chave, à esquerda): AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, DATA_BUCKET
from getpass import getpass

for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    try:
        os.environ[k] = userdata.get(k).strip()  # Secrets do Colab (no browser)
    except Exception:
        os.environ[k] = getpass(f"{k}: ").strip()  # noutro sítio: pede o valor sem o mostrar
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets boto3 pydantic

from google.colab import drive, userdata

drive.mount("/content/drive")          # os checkpoints vão para o Drive: sobrevivem a desconexões
RUN_DIR = "/content/drive/MyDrive/wiki-bio-distill/run-v1"

!git clone https://github.com/pinthoz/wiki-bio-distill.git /content/repo
%cd /content/repo
!mkdir -p data preds
!for f in train val test; do aws s3 cp s3://$DATA_BUCKET/v1/$f.jsonl data/; done

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo
download: s3://pt-bio-extract-data-ffe8781fe9cd56e1f937d41279/v1/train.jsonl to data/train.jsonl
download: s3://pt-bio-extract-data-ffe8781fe9cd56e1f937d41279/v1/val.jsonl to data/val.jsonl
download: s3://pt-bio-extract-data-ffe8781fe9cd56e1f937d41279/v1/test.jsonl to data/test.jsonl


In [9]:
%cd /content/repo
!git pull
!ls gen.py evaluate.py

import importlib, sys
importlib.invalidate_caches()
if "/content/repo" not in sys.path:
    sys.path.insert(0, "/content/repo")


/content/repo
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 10 (delta 3), reused 10 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 8.67 KiB | 1.73 MiB/s, done.
From https://github.com/pinthoz/wiki-bio-distill
   c114c89..0407dbd  main       -> origin/main
Updating c114c89..0407dbd
Fast-forward
 evaluate.py                         |  55 +++
 gen.py                              |  45 +++
 notebook/01_label_teacher.ipynb     | 358 +++++++++----------
 notebook/02_train_eval_export.ipynb | 682 ++++++++++++++++++------------------
 notebook/train.ipynb                | 119 +++++++
 review.py                           |  95 +++++
 split.py                            |  75 ++++
 7 files changed, 909 insertions(+), 520 deletions(-)
 create mode 100644 gen.py
evaluate.py  gen.py


In [10]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from common import SYSTEM_TEACHER
from gen import generate_preds

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")

test = [json.loads(l) for l in open("data/test.jsonl")]
# O base recebe o prompt detalhado: é o melhor que consegue sem treino
generate_preds(base, tokenizer, test, SYSTEM_TEACHER, "preds/base.jsonl")
!python evaluate.py preds/base.jsonl

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

8/150
16/150
24/150
32/150
40/150
48/150
56/150
64/150
72/150
80/150
88/150
96/150
104/150
112/150
120/150
128/150
136/150
144/150
150/150
métrica                    professor          base
JSON válido                    1.000         0.653
F1 global                      1.000         0.635
precisão global                1.000         0.841
recall global                  0.999         0.510
F1 name                        1.000         0.653
F1 birth_date                  1.000         0.793
F1 birth_place                 1.000         0.661
F1 death_date                  1.000         0.440
F1 death_place                 1.000         0.391
F1 nationality                 0.996         0.652
F1 occupations                 1.000         0.603
latência base: p50=660 ms  p95=814 ms


In [13]:
import gc, glob, torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# Free the GPU if the baseline model is still loaded
if "base" in globals():
    del base
gc.collect(); torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    dtype=torch.float16,  # FIX: the T4 has no bf16, keep the non-quantized layers in fp16
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)
tokenizer.padding_side = "right"  # gen.py sets "left" for generation; training uses right
data = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})

peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")
args = SFTConfig(
    output_dir=RUN_DIR, num_train_epochs=2,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10,
    max_length=1024, fp16=True, gradient_checkpointing=True,
    logging_steps=10, eval_strategy="steps", eval_steps=50, save_steps=50, save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"],
                     eval_dataset=data["validation"], processing_class=tokenizer,
                     peft_config=peft_config)

# FIX: LoRA weights in fp32, as fp16 mixed precision expects
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
trainer.model.print_trainable_parameters()

# If Colab disconnected mid-run, resume from the last checkpoint on Drive
resume = bool(glob.glob(f"{RUN_DIR}/checkpoint-*"))
trainer.train(resume_from_checkpoint=resume)

trainer.save_model(f"{RUN_DIR}/adapter")
tokenizer.save_pretrained(f"{RUN_DIR}/adapter")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.037336,0.023436,0.030287,154304.000000,0.992934
100,0.024656,0.021364,0.027381,311372.000000,0.993748
150,0.027667,0.019255,0.021903,469745.000000,0.994585
200,0.013611,0.018076,0.014839,624078.000000,0.994011
250,0.015354,0.016397,0.014987,781319.000000,0.994595
300,0.009741,0.016180,0.014503,937901.000000,0.994766
338,0.012387,0.016105,0.014318,1053324.000000,0.994766


('/content/drive/MyDrive/wiki-bio-distill/run-v1/adapter/tokenizer_config.json',
 '/content/drive/MyDrive/wiki-bio-distill/run-v1/adapter/chat_template.jinja',
 '/content/drive/MyDrive/wiki-bio-distill/run-v1/adapter/tokenizer.json')

In [18]:
DRIVE = "/content/drive/MyDrive/wiki-bio-distill"
RUN_DIR = f"{DRIVE}/run-v1"
!ls {RUN_DIR}
!ls {RUN_DIR}/adapter


adapter  checkpoint-250  checkpoint-300  checkpoint-338  README.md
adapter_config.json	   README.md		  training_args.bin
adapter_model.safetensors  tokenizer_config.json
chat_template.jinja	   tokenizer.json


In [19]:
!aws s3 cp --recursive {RUN_DIR}/adapter s3://$$DATA_BUCKET/v1/models/qwen1.5b-lora-r16/adapter/



The user-provided path {RUN_DIR}/adapter does not exist.
